# 1.14 LangChain · LangGraph · CrewAI
### Frameworks de Agentes IA en Google Colab

---

| Framework | Abstraccion | Mejor para |
|-----------|-------------|-----------|
| **LangChain** | Media | Cadenas, RAG, conectores |
| **LangGraph** | Baja (mas control) | Agentes con estado, ciclos, flujos complejos |
| **CrewAI** | Alta (declarativo) | Multi-agente con roles, pipelines de produccion |

**Contenidos:**
1. Instalacion y setup
2. LangChain: LCEL, cadenas y prompts
3. LangChain: Agentes con herramientas
4. LangChain: Memoria y RAG
5. LangGraph: Grafos de estado
6. LangGraph: Agente ReAct como grafo
7. LangGraph: Multi-agente con grafo
8. CrewAI: Agentes, tareas y crews
9. CrewAI: Herramientas personalizadas
10. CrewAI: Proceso jerarquico
11. Comparativa final

> **Setup:** activa GPU T4 en `Editar > Configuracion del cuaderno`.

---
# PARTE 1 — Instalacion y Setup

In [26]:
# LangChain ecosystem
# langchain-openai    -> integracion con OpenAI (ChatOpenAI, Embeddings)
# langchain-community -> conectores: FAISS, loaders, etc.
# langgraph           -> agentes como grafos de estado
# faiss-cpu           -> busqueda vectorial rapida en CPU

!pip install langchain langchain-openai langchain-community langgraph faiss-cpu tiktoken -q

# CrewAI: framework de multi-agente declarativo
!pip install crewai crewai-tools -q

print("Instalacion completa.")

Instalacion completa.


In [ ]:
import os, json, time
from IPython.display import display

# ── API KEY ─────────────────────────────────────────────────────
try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    print("API key cargada desde Colab Secrets.")
except Exception:
    os.environ['OPENAI_API_KEY'] = os.environ.get('OPENAI_API_KEY', 'Api_Key')
    print("API key cargada desde variable de entorno.")

# Colores para distinguir los frameworks
LC  = '[94m'; LG = '[92m'; CA = '[95m'
RST = '[0m';  BOLD = '[1m'

def titulo(texto, color=None):
    c = color or LC
    print(f"\n{c}{BOLD}{'='*60}\n  {texto}\n{'='*60}{RST}")

print("Setup listo.")

API key cargada desde variable de entorno.
Setup listo.


---
# PARTE 2 — LangChain: LCEL, Cadenas y Prompts

**LangChain Expression Language (LCEL)** usa el operador `|` para encadenar componentes Runnable:

```
prompt | llm | output_parser
```

Cada componente recibe una entrada, la transforma y la pasa al siguiente.
LCEL soporta ejecucion paralela, streaming y trazabilidad automatica.

In [28]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.messages import HumanMessage, AIMessage
from pydantic import BaseModel, Field
from typing import List

titulo("LangChain -- Cadena LCEL basica", LC)

# ChatOpenAI: wrapper de LangChain sobre la API de OpenAI
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

# ChatPromptTemplate: define la estructura del prompt con variables entre {}
# Las variables se rellenan al invocar la cadena con .invoke({...})
prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un experto en {dominio}. Responde en espanol de forma concisa."),
    ("human",  "{pregunta}")
])

# StrOutputParser: extrae el texto plano de la respuesta AIMessage
# Sin parser: obtienes un objeto AIMessage
# Con parser: obtienes directamente un string
parser = StrOutputParser()

# La cadena LCEL: prompt -> llm -> parser
# El operador | conecta Runnables en secuencia
cadena = prompt | llm | parser

# .invoke(): ejecuta la cadena con un diccionario de variables
respuesta = cadena.invoke({
    "dominio":  "inteligencia artificial y frameworks de LLMs",
    "pregunta": "Explica que es LangChain en 2 oraciones."
})
print(f"{LC}Respuesta:{RST}\n{respuesta}")


  LangChain -- Cadena LCEL basica
Respuesta:
LangChain es un framework diseñado para facilitar la construcción de aplicaciones que integran modelos de lenguaje, permitiendo la conexión de estos con diversas fuentes de datos y servicios. Proporciona herramientas y componentes que simplifican la gestión de flujos de trabajo y la interacción con APIs, mejorando la funcionalidad y la flexibilidad de las aplicaciones basadas en inteligencia artificial.


In [29]:
# ── SALIDA JSON ESTRUCTURADA ────────────────────────────────────
# JsonOutputParser fuerza al LLM a responder en un esquema definido con Pydantic.
# LangChain genera automaticamente las instrucciones de formato para el LLM.

titulo("LCEL con salida JSON (Pydantic)", LC)

class AnalisisFramework(BaseModel):
    nombre      : str       = Field(description="Nombre del framework")
    ventajas    : List[str] = Field(description="Lista de ventajas principales (max 3)")
    desventajas : List[str] = Field(description="Lista de desventajas (max 3)")
    casos_uso   : List[str] = Field(description="Casos de uso recomendados (max 3)")
    puntuacion  : int       = Field(description="Puntuacion de 1 a 10 para agentes de produccion")

json_parser = JsonOutputParser(pydantic_object=AnalisisFramework)

# .partial(): pre-rellena variables del prompt que no cambian entre llamadas
prompt_json = ChatPromptTemplate.from_messages([
    ("system", "Analiza frameworks de IA con precision.\n{format_instructions}"),
    ("human",  "Analiza el framework: {framework}")
]).partial(format_instructions=json_parser.get_format_instructions())

cadena_json = prompt_json | llm | json_parser

for fw in ["LangChain", "LangGraph", "CrewAI"]:
    res = cadena_json.invoke({"framework": fw})
    print(f"\n{BOLD}{res['nombre']}{RST} (puntuacion: {res['puntuacion']}/10)")
    print(f"  Ventajas : {res['ventajas']}")
    print(f"  Uso ideal: {res['casos_uso']}")


  LCEL con salida JSON (Pydantic)

LangChain (puntuacion: 8/10)
  Ventajas : ['Facilita la creación de aplicaciones complejas de IA que utilizan múltiples modelos.', 'Proporciona herramientas integradas para la gestión de datos y flujos de trabajo.', 'Compatibilidad con múltiples proveedores de modelos de lenguaje.']
  Uso ideal: ['Desarrollo de chatbots avanzados.', 'Creación de sistemas de recomendación personalizados.', 'Automatización de tareas de procesamiento de lenguaje natural.']

LangGraph (puntuacion: 7/10)
  Ventajas : ['Integración fluida con múltiples lenguajes de programación', 'Facilita la creación de flujos de trabajo complejos', 'Optimización en la gestión de recursos']
  Uso ideal: ['Desarrollo de aplicaciones de procesamiento de lenguaje natural', 'Automatización de tareas mediante flujos de trabajo', 'Análisis de datos y visualización']

CrewAI (puntuacion: 7/10)
  Ventajas : ['Interfaz fácil de usar', 'Integración con múltiples plataformas', 'Optimización de proce

In [30]:
# ── CADENA SECUENCIAL: generar -> criticar -> mejorar ───────────
titulo("Cadena secuencial: borrador -> critica -> version mejorada", LC)

llm2 = ChatOpenAI(model="gpt-4o-mini", temperature=0.5)

generar   = ChatPromptTemplate.from_messages([
    ("system", "Redactor tecnico. Escribe claro y conciso."),
    ("human",  "Escribe un parrafo sobre: {tema}")
]) | llm2 | StrOutputParser()

criticar  = ChatPromptTemplate.from_messages([
    ("system", "Editor critico. Identifica exactamente 2 mejoras especificas."),
    ("human",  "Critica este texto:\n{texto}")
]) | llm2 | StrOutputParser()

mejorar   = ChatPromptTemplate.from_messages([
    ("system", "Redactor experto. Mejora el texto aplicando la critica."),
    ("human",  "Texto: {texto}\n\nCritica: {critica}\n\nVersion mejorada:")
]) | llm2 | StrOutputParser()

tema = "por que LangGraph supera a las cadenas lineales para agentes complejos"

borrador = generar.invoke({"tema": tema})
print(f"{BOLD}Borrador:{RST}\n{borrador}\n")

critica  = criticar.invoke({"texto": borrador})
print(f"{BOLD}Critica:{RST}\n{critica}\n")

mejorado = mejorar.invoke({"texto": borrador, "critica": critica})
print(f"{BOLD}Version mejorada:{RST}\n{mejorado}")


  Cadena secuencial: borrador -> critica -> version mejorada
Borrador:
LangGraph supera a las cadenas lineales en la gestión de agentes complejos debido a su capacidad para representar relaciones no lineales y dinámicas entre múltiples componentes. A diferencia de las cadenas lineales, que siguen una secuencia rígida y unidimensional, LangGraph permite la interconexión de nodos que pueden interactuar de manera más flexible y contextual. Esto facilita un procesamiento más eficiente de la información, mejorando la toma de decisiones y la adaptabilidad en entornos complejos, donde las interacciones entre agentes son multifacéticas y requieren un enfoque más holístico.

Critica:
1. **Claridad y simplicidad**: El texto utiliza términos técnicos como "cadenas lineales" y "nodos" sin una explicación previa. Para mejorar la comprensión, sería útil ofrecer una breve definición de estos conceptos o ejemplos concretos que ilustren cómo funcionan, especialmente para lectores que no están familiar

---
# PARTE 3 — LangGraph: Grafos de Estado

LangGraph modela el agente como un **grafo dirigido**:
- **Nodos**: funciones Python que transforman el estado.
- **Edges**: transiciones entre nodos (fijas o condicionales).
- **Estado**: `TypedDict` tipado que fluye por todos los nodos.

La clave diferenciadora: soporta **ciclos** (loops) para reintentos, reflexion y flujos iterativos.

In [52]:
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage, AIMessage
from typing import TypedDict, Annotated
import operator

titulo("LangGraph -- StateGraph: nodos, edges y ciclos", LG)

# ESTADO DEL GRAFO
# TypedDict define los campos del estado compartido entre todos los nodos.
# Annotated[List, operator.add]: el campo se ACUMULA (append) en cada actualizacion.
# Sin Annotated: el campo se SOBREESCRIBE en cada actualizacion.
class EstadoRevision(TypedDict):
    tema     : str                               # tema de trabajo
    texto    : str                               # texto actual
    critica  : str                               # critica recibida
    version  : int                               # contador de intentos
    aprobado : bool                              # condicion de salida
    log      : Annotated[list, operator.add]     # log acumulativo

llm_g = ChatOpenAI(model="gpt-4o-mini", temperature=0.6)

# NODOS: funciones Python que reciben el estado y retornan un dict PARCIAL
# Solo necesitas incluir los campos que cambiaron

def generar_texto(estado: EstadoRevision) -> dict:
    "Genera o mejora el texto segun la critica previa."
    instruccion = f"Escribe un parrafo tecnico sobre: {estado['tema']}. Version {estado['version']}."
    if estado['critica']:
        instruccion += f" Considerando esta critica: {estado['critica']}"
    resp = llm_g.invoke([HumanMessage(content=instruccion)])
    print(f"  {LG}[Generar v{estado['version']}]{RST} {resp.content[:80]}...")
    return {
        "texto": resp.content,
        "log":   [f"v{estado['version']}: texto generado"],
    }

def evaluar_texto(estado: EstadoRevision) -> dict:
    "Evalua si el texto cumple el estandar de calidad."
    resp = llm_g.invoke([HumanMessage(content=
        f"Evalua este texto. Responde solo: APROBADO o RECHAZADO: <razon breve>.\n"
        f"Texto: {estado['texto']}"
    )])
    critica  = resp.content.strip()
    aprobado = critica.upper().startswith("APROBADO")
    print(f"  {LG}[Evaluar]{RST} {'Aprobado' if aprobado else 'Rechazado'}: {critica[:60]}")
    return {"critica": critica, "aprobado": aprobado, "log": [f"evaluado: {'ok' if aprobado else 'rechazado'}"]}

def finalizar(estado: EstadoRevision) -> dict:
    "Nodo final: el texto paso la evaluacion."
    print(f"  {LG}[Finalizar]{RST} Texto aprobado en version {estado['version']}.")
    return {"log": ["FINALIZADO"]}

def incrementar(estado: EstadoRevision) -> dict:
    "Incrementa el contador de version para el siguiente ciclo."
    return {"version": estado['version'] + 1, "log": [f"reintento -> v{estado['version']+1}"]}

# EDGE CONDICIONAL: decide el siguiente nodo segun el estado
def router(estado: EstadoRevision) -> str:
    if estado["aprobado"]:       return "finalizar"   # salir del ciclo
    elif estado["version"] < 3:  return "incrementar" # ciclo: reintentar
    else:                        return "finalizar"    # forzar fin despues de 3 intentos

# CONSTRUIR EL GRAFO
g = StateGraph(EstadoRevision)
g.add_node("generar",    generar_texto)
g.add_node("evaluar",    evaluar_texto)
g.add_node("finalizar",  finalizar)
g.add_node("incrementar",incrementar)

g.set_entry_point("generar")                          # nodo de entrada
g.add_edge("generar",    "evaluar")                   # edge fijo
g.add_edge("finalizar",  END)                         # edge al fin
g.add_edge("incrementar","generar")                   # ciclo: volver a generar

# add_conditional_edges: routing segun el valor retornado por 'router'
g.add_conditional_edges("evaluar", router,
    {"finalizar":"finalizar", "incrementar":"incrementar"})

app_g = g.compile()

print("\nEjecutando grafo con ciclo de revision:\n")
res = app_g.invoke({
    "tema": "por que LangGraph soporta ciclos mientras las cadenas lineales no pueden",
    "texto": "", "critica": "", "version": 1, "aprobado": False, "log": []
})
print(f"\n{LG}Texto final (version {res['version']}):{RST}")
print(res['texto'])
print(f"\nLog del proceso: {res['log']}")


  LangGraph -- StateGraph: nodos, edges y ciclos

Ejecutando grafo con ciclo de revision:

  [Generar v1] LangGraph, a diferencia de las cadenas lineales, está diseñado para manejar estr...
  [Evaluar] Aprobado: APROBADO: El texto presenta una explicación clara y coherent
  [Finalizar] Texto aprobado en version 1.

Texto final (version 1):
LangGraph, a diferencia de las cadenas lineales, está diseñado para manejar estructuras de datos que permiten ciclos, lo que facilita la representación de relaciones complejas y recursivas entre nodos. En una cadena lineal, cada elemento está secuencialmente conectado a su predecesor y sucesor, lo que impide la formación de ciclos, ya que cualquier conexión que regrese a un nodo anterior rompería la secuencia lineal. En contraste, LangGraph utiliza una estructura de grafo donde los nodos pueden referenciarse entre sí de manera no lineal, permitiendo que un nodo apunte a uno de sus predecesores o a otros nodos en el grafo, creando así ciclos. Esta ca

In [51]:
titulo("LangGraph -- Agente ReAct prebuilt", LG)

# create_agent construye el grafo ReAct internamente:
# [LLM] -> (si tool_calls) -> [Tools] -> [LLM] -> ... -> respuesta
# Es el patron ReAct implementado como grafo de LangGraph

llm_react = ChatOpenAI(model="gpt-4o-mini", temperature=0)

agente_react = create_agent(
    model=llm_react,
    tools=herramientas_lc
)

res_react = agente_react.invoke({
    "messages": [HumanMessage(content=
        "Busca informacion sobre RAG y LangGraph. "
        "Luego ejecuta: print([x**2 for x in range(1,6)]). "
        "Finalmente calcula la suma de los cuadrados del 1 al 5."
    )]
})

print(f"{LG}Respuesta:{RST} {res_react['messages'][-1].content}")
print(f"\nMensajes en el grafo: {len(res_react['messages'])}")
for m in res_react['messages']:
    tipo = type(m).__name__
    cont = str(m.content)[:70] + "..." if len(str(m.content)) > 70 else str(m.content)
    print(f"  {tipo}: {cont}")


  LangGraph -- Agente ReAct prebuilt


KeyboardInterrupt: 

In [ ]:
titulo("LangChain -- Definicion de herramientas con @tool", LC)

@tool
def buscar_informacion(query: str) -> str:
    '''Busca informacion sobre frameworks de IA: LangChain, LangGraph, CrewAI, RAG, agentes.'''
    base = {
        "langchain" : "LangChain: framework modular para LLMs. LCEL con |. 80K+ GitHub stars. Creado por Harrison Chase, 2022.",
        "langgraph" : "LangGraph: extension de LangChain para agentes con estado como grafos. StateGraph + TypedDict. Soporta ciclos.",
        "crewai"    : "CrewAI: multi-agente declarativo. Agent + Task + Crew. Process.sequential/hierarchical. Lanzado 2024.",
        "rag"       : "RAG: Retrieval Augmented Generation. Busqueda vectorial + LLM para respuestas con informacion actualizada.",
        "agente"    : "Agente de IA: entidad que percibe el entorno y actua. Bucle percepcion-razonamiento-accion.",
        "react"     : "ReAct: Pensamiento + Accion + Observacion. Entrelaza razonamiento y accion. Yao et al. 2022.",
    }
    for k, v in base.items():
        if k in query.lower():
            return v
    return f"Informacion sobre '{query}': tema del ecosistema de LLMs y agentes de IA."


@tool
def calcular(expresion: str) -> str:
    '''Evalua expresiones matematicas. Acepta +, -, *, /, **, %. Ejemplo: '2**10 + 50/2'.'''
    import ast
    try:
        nodos = {ast.Expression, ast.Constant, ast.BinOp, ast.UnaryOp,
                 ast.Add, ast.Sub, ast.Mult, ast.Div, ast.Pow, ast.Mod, ast.USub}
        tree  = ast.parse(expresion, mode='eval')
        for n in ast.walk(tree):
            if type(n) not in nodos:
                return "Expresion no permitida."
        return str(round(eval(compile(tree, '', 'eval')), 6))
    except Exception as e:
        return f"Error: {e}"


@tool
def ejecutar_python(codigo: str) -> str:
    '''Ejecuta codigo Python y retorna la salida del print(). Usa para logica, listas, calculos complejos.'''
    import io, contextlib
    buf = io.StringIO()
    builtins_ok = {"print":print,"range":range,"len":len,"sum":sum,"max":max,
                   "min":min,"sorted":sorted,"enumerate":enumerate,"list":list,
                   "dict":dict,"str":str,"int":int,"float":float,"round":round,"abs":abs}
    try:
        with contextlib.redirect_stdout(buf):
            exec(codigo, {"__builtins__": builtins_ok})
        return buf.getvalue() or "Ejecutado sin salida."
    except Exception as e:
        return f"Error: {e}"


herramientas_lc = [buscar_informacion, calcular, ejecutar_python]
print(f"Herramientas: {[h.name for h in herramientas_lc]}")
print(f"\nEsquema de 'calcular':")
print(json.dumps(calcular.args_schema.schema(), indent=2))

In [36]:
from langgraph.checkpoint.memory import MemorySaver

titulo("LangGraph -- Agente con memoria persistente entre sesiones", LG)

# MemorySaver: guarda el estado del grafo en memoria por thread_id
# En produccion: SqliteSaver, PostgresSaver, RedisSaver
memory = MemorySaver()

agente_persistente = create_agent(
    model=ChatOpenAI(model="gpt-4o-mini", temperature=0),
    tools=herramientas_lc
)

# thread_id: identificador de la sesion
# Conversaciones distintas = thread_ids distintos
config = {"configurable": {"thread_id": "sesion-demo-001"}}

print("Conversacion multi-turno con estado persistente:\n")

r1 = agente_persistente.invoke(
    {"messages": [HumanMessage(content="Mi proyecto usa LangGraph y se llama NexusAgent.")]},
    config=config
)
print(f"  {BOLD}[T1]{RST} {r1['messages'][-1].content[:150]}\n")

# El agente recuerda el turno anterior sin que lo repitamos
r2 = agente_persistente.invoke(
    {"messages": [HumanMessage(content="Que framework usa mi proyecto y cuantas letras tiene su nombre?")]},
    config=config
)
print(f"  {BOLD}[T2]{RST} {r2['messages'][-1].content[:200]}\n")

r3 = agente_persistente.invoke(
    {"messages": [HumanMessage(content="Busca info sobre el framework de mi proyecto.")]},
    config=config
)
print(f"  {BOLD}[T3]{RST} {r3['messages'][-1].content[:200]}")


  LangGraph -- Agente con memoria persistente entre sesiones
Conversacion multi-turno con estado persistente:



/tmp/ipykernel_3096/2898729998.py:9: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agente_persistente = create_react_agent(


  [T1] LangGraph es un framework que permite representar agentes como grafos de estado, utilizando una combinación de StateGraph y TypedDict. Este enfoque so

  [T2] Tu proyecto, NexusAgent, utiliza el framework LangGraph. El nombre "LangGraph" tiene 9 letras.

  [T3] LangGraph es un framework que permite representar agentes como grafos de estado. Combina StateGraph y TypedDict, y tiene la capacidad de soportar ciclos. Esto lo hace adecuado para modelar comportamie


---
# PARTE 7 — LangGraph: Sistema Multi-Agente

En LangGraph, los sistemas multi-agente se modelan como grafos donde **cada nodo puede ser un agente completo**. Un nodo orquestador decide a qué especialista enrutar en cada momento.

In [37]:
titulo("LangGraph -- Multi-agente con grafo de orquestacion", LG)

class EstadoMultiAg(TypedDict):
    tarea           : str
    siguiente       : str
    res_investigador: str
    res_analista    : str
    respuesta_final : str
    log             : Annotated[list, operator.add]

llm_ma = ChatOpenAI(model="gpt-4o-mini", temperature=0.4)

def orquestador(estado: EstadoMultiAg) -> dict:
    "Decide que agente activar segun el progreso actual."
    tiene_investigacion = bool(estado.get("res_investigador"))
    tiene_analisis      = bool(estado.get("res_analista"))

    decision = llm_ma.invoke([HumanMessage(content=
        f"Tarea: {estado['tarea']}\n"
        f"Investigacion hecha: {'SI' if tiene_investigacion else 'NO'}\n"
        f"Analisis hecho: {'SI' if tiene_analisis else 'NO'}\n"
        "Responde SOLO con una de estas palabras:\n"
        "INVESTIGAR | ANALIZAR | FINALIZAR"
    )])
    sig = "investigador" if "INVESTIGAR" in decision.content.upper() else           "analista"     if "ANALIZAR"   in decision.content.upper() else "sintetizador"
    print(f"  {LG}[Orquestador]{RST} -> {sig}")
    return {"siguiente": sig, "log": [f"orquestador -> {sig}"]}

def nodo_investigador(estado: EstadoMultiAg) -> dict:
    resp = llm_ma.invoke([HumanMessage(content=
        f"Eres un investigador. Responde con hechos concretos sobre: {estado['tarea']}"
    )])
    print(f"  {LG}[Investigador]{RST} {resp.content[:80]}...")
    return {"res_investigador": resp.content, "log": ["investigador completado"]}

def nodo_analista(estado: EstadoMultiAg) -> dict:
    resp = llm_ma.invoke([HumanMessage(content=
        f"Eres un analista. Basandote en esto:\n{estado['res_investigador']}\n"
        f"Extrae los datos clave para: {estado['tarea']}"
    )])
    print(f"  {LG}[Analista]{RST} {resp.content[:80]}...")
    return {"res_analista": resp.content, "log": ["analista completado"]}

def nodo_sintetizador(estado: EstadoMultiAg) -> dict:
    ctx  = f"Investigacion: {estado.get('res_investigador','')}\nAnalisis: {estado.get('res_analista','')}"
    resp = llm_ma.invoke([HumanMessage(content=
        f"Sintetiza una respuesta completa para: {estado['tarea']}\n\n{ctx}"
    )])
    print(f"  {LG}[Sintetizador]{RST} Respuesta final generada.")
    return {"respuesta_final": resp.content, "log": ["sintetizador completado"]}

def routing(estado: EstadoMultiAg) -> str:
    return estado["siguiente"]

# Construir grafo multi-agente
gma = StateGraph(EstadoMultiAg)
gma.add_node("orquestador",   orquestador)
gma.add_node("investigador",  nodo_investigador)
gma.add_node("analista",      nodo_analista)
gma.add_node("sintetizador",  nodo_sintetizador)

gma.set_entry_point("orquestador")
gma.add_conditional_edges("orquestador", routing,
    {"investigador":"investigador","analista":"analista","sintetizador":"sintetizador"})
gma.add_edge("investigador", "orquestador")   # despues de investigar -> volver al orquestador
gma.add_edge("analista",     "orquestador")   # despues de analizar   -> volver al orquestador
gma.add_edge("sintetizador", END)

app_ma = gma.compile()

print("Ejecutando sistema multi-agente con LangGraph:\n")
res_ma = app_ma.invoke({
    "tarea"           : "Compara LangGraph y CrewAI para sistemas multi-agente en produccion",
    "siguiente"       : "",
    "res_investigador": "",
    "res_analista"    : "",
    "respuesta_final" : "",
    "log"             : [],
})
print(f"\n{LG}Respuesta final:{RST}\n{res_ma['respuesta_final']}")
print(f"\nLog: {res_ma['log']}")


  LangGraph -- Multi-agente con grafo de orquestacion
Ejecutando sistema multi-agente con LangGraph:

  [Orquestador] -> investigador
  [Investigador] LangGraph y CrewAI son dos enfoques diferentes para el desarrollo de sistemas mu...
  [Orquestador] -> analista
  [Analista] Aquí tienes una comparación de LangGraph y CrewAI para sistemas multi-agente en ...
  [Orquestador] -> sintetizador
  [Sintetizador] Respuesta final generada.

Respuesta final:
### Comparación de LangGraph y CrewAI para Sistemas Multi-Agente en Producción

LangGraph y CrewAI son dos enfoques distintos para el desarrollo de sistemas multi-agente, cada uno con características y aplicaciones específicas. A continuación, se presenta una comparación basada en aspectos clave:

#### LangGraph

1. **Modelo de Lenguaje**: Utiliza modelos de lenguaje y técnicas de procesamiento de lenguaje natural (NLP) para facilitar la interacción entre agentes y con usuarios humanos.
   
2. **Interoperabilidad**: Diseñado para integrarse

---
# PARTE 8 — CrewAI: Agentes, Tareas y Crews

CrewAI es el framework mas **declarativo**. Defines quien es cada agente (`Agent`), que debe hacer (`Task`), y como trabajan juntos (`Crew`).

El framework se encarga del bucle de ejecucion, delegacion y contexto entre tareas.

In [40]:
from crewai import Agent, Task, Crew, Process
from langchain_openai import ChatOpenAI # Reverted import based on deprecation warning

titulo("CrewAI -- Definicion de Agentes", CA)

# Keep llm_ca defined for consistency, but use model name string for agent llm
llm_model_name = "gpt-4o-mini" # Define model name as string
llm_temperature = 0.7 # Store temperature separately if needed later for crew config

# Agent: define la identidad del agente
# - role     : titulo que influye en el comportamiento del LLM
# - goal     : objetivo principal del agente
# - backstory: contexto que da personalidad y expertise
# - llm      : modelo de lenguaje que usa este agente
# - verbose  : mostrar el proceso de razonamiento interno

investigador_ca = Agent(
    role="Investigador Senior de Frameworks de IA",
    goal="Investigar y documentar con precision las caracteristicas tecnicas de frameworks de IA.",
    backstory=(
        "Investigador con 10 anos de experiencia en el ecosistema de LLMs. "
        "Has contribuido a la documentacion oficial de LangChain y LangGraph. "
        "Eres conocido por tu rigor tecnico y capacidad de sintesis."
    ),
    llm=llm_model_name, # Pass model name as string to satisfy validation
    verbose=True,
    allow_delegation=False,
)

analista_ca = Agent(
    role="Analista de Arquitecturas de Software",
    goal="Comparar frameworks de IA y generar recomendaciones arquitectonicas basadas en requisitos.",
    backstory=(
        "Arquitecto de software con experiencia en sistemas de IA a escala. "
        "Has implementado pipelines de agentes en produccion para empresas Fortune 500. "
        "Tu fortaleza es identificar el framework correcto para cada caso de uso."
    ),
    llm=llm_model_name, # Pass model name as string to satisfy validation
    verbose=True,
    allow_delegation=False,
)

redactor_ca = Agent(
    role="Redactor Tecnico Senior",
    goal="Transformar analisis tecnicos en documentos ejecutivos claros y accionables.",
    backstory=(
        "Redactor con experiencia en documentacion tecnica de proyectos de IA. "
        "Has escrito documentacion para frameworks usados por miles de desarrolladores. "
        "Sabes estructurar informacion para audiencias tecnicas y ejecutivas."
    ),
    llm=llm_model_name, # Pass model name as string to satisfy validation
    verbose=True,
    allow_delegation=True,   # puede pedir informacion adicional al investigador
)

print("Agentes CrewAI creados:")
for ag in [investigador_ca, analista_ca, redactor_ca]:
    print(f"  [{ag.role}] goal: {ag.goal[:60]}...")


  CrewAI -- Definicion de Agentes
Agentes CrewAI creados:
  [Investigador Senior de Frameworks de IA] goal: Investigar y documentar con precision las caracteristicas te...
  [Analista de Arquitecturas de Software] goal: Comparar frameworks de IA y generar recomendaciones arquitec...
  [Redactor Tecnico Senior] goal: Transformar analisis tecnicos en documentos ejecutivos claro...


In [41]:
titulo("CrewAI -- Definicion de Tareas y Crew secuencial", CA)

# Task: define que debe hacer un agente
# - description   : que hacer exactamente (el LLM lee esto como instrucciones)
# - expected_output: como debe ser la salida (el LLM lo usa como criterio de exito)
# - agent         : quien hace la tarea
# - context       : lista de tareas previas cuya salida se inyecta como contexto

tarea_investigar = Task(
    description=(
        "Investiga los tres frameworks: LangChain, LangGraph y CrewAI. "
        "Para cada uno documenta: (1) proposito principal, (2) componentes clave, "
        "(3) modelo de ejecucion, (4) casos de uso ideales. "
        "Presenta la informacion de forma estructurada y tecnica."
    ),
    expected_output=(
        "Un informe con una seccion por framework (titulo en mayusculas), "
        "cada seccion con los 4 puntos numerados. Maximo 150 palabras por framework."
    ),
    agent=investigador_ca,
)

tarea_comparar = Task(
    description=(
        "Basandote en la investigacion previa, realiza un analisis comparativo. "
        "Crea una tabla comparativa con 6 criterios: curva de aprendizaje, "
        "control del flujo, soporte multi-agente, RAG, estado persistente, produccion. "
        "Añade una recomendacion para cada tipo de proyecto."
    ),
    expected_output=(
        "Tabla comparativa en formato texto ASCII con los 6 criterios. "
        "Recomendaciones concretas: (1) para RAG, (2) para agente individual, (3) para multi-agente."
    ),
    agent=analista_ca,
    context=[tarea_investigar],   # usa la salida de investigar como contexto
)

tarea_informe = Task(
    description=(
        "Redacta un informe ejecutivo integrando la investigacion y el analisis. "
        "El informe va dirigido a un CTO tecnico que debe elegir un framework. "
        "Estructura: (1) resumen de 3 puntos clave, (2) comparativa resumida, "
        "(3) recomendacion final justificada con el framework ganador y por que."
    ),
    expected_output=(
        "Informe ejecutivo de 350-450 palabras con los 3 apartados estructurados, "
        "tono profesional y tecnico, con una recomendacion final clara."
    ),
    agent=redactor_ca,
    context=[tarea_investigar, tarea_comparar],
)

# Crew: el equipo completo
# Process.sequential: tarea1 -> tarea2 -> tarea3 (en orden, contexto encadenado)
crew_ca = Crew(
    agents=[investigador_ca, analista_ca, redactor_ca],
    tasks=[tarea_investigar, tarea_comparar, tarea_informe],
    process=Process.sequential,
    verbose=True,
)

print("Ejecutando crew secuencial... (puede tardar 1-2 minutos)\n")
resultado_ca = crew_ca.kickoff()
print(f"\n{CA}{BOLD}RESULTADO FINAL DEL CREW:{RST}")
print(resultado_ca)


  CrewAI -- Definicion de Tareas y Crew secuencial
Ejecutando crew secuencial... (puede tardar 1-2 minutos)



╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 804f125c-cd19-426e-abc1-74a183862af7                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Investiga los tres frameworks: LangChain, LangGraph y CrewAI. Para cada uno documenta: (1) proposito     │
│  principal, (2) componentes clave, (3) modelo de ejecucion, (4) casos de uso ideales. Presenta la informacion   │
│  de forma estructurada y tecnica.                                                                               │
│  ID: 0fd7a3e3-cf39-41a7-823a-5f4773b99881                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Investigador Senior de Frameworks de IA                                                                 │
│                                                                                                                 │
│  Task: Investiga los tres frameworks: LangChain, LangGraph y CrewAI. Para cada uno documenta: (1) proposito     │
│  principal, (2) componentes clave, (3) modelo de ejecucion, (4) casos de uso ideales. Presenta la informacion   │
│  de forma estructurada y tecnica.                                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Investigador Senior de Frameworks de IA                                                                 │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # LANGCHAIN                                                                                                    │
│                                                                                                                 │
│  1. **Propósito principal**: LangChain está diseñado para facilitar la creación de aplicaciones que integran    │
│  LLMs (Large Language Models) con otras fuentes de datos mediante una arquitectura modular.                     │
│                                                                                                                 │
│  2. **Componentes clave**: Incluye cadenas, agentes, memoria y herramientas. Las cadenas permiten secuenciar    │
│  llamadas a LLMs, mientras que los agentes permiten decisiones dinámicas. La memoria guarda el contexto, y las  │
│  herramientas permiten la integración con APIs externas.                                                        │
│                                                                                                                 │
│  3. **Modelo de ejecución**: LangChain opera en un modelo de ejecución basado en flujo, donde cada paso puede   │
│  depender de los resultados de los anteriores, permitiendo una interacción fluida y coherente con el LLM.       │
│                                                                                                                 │
│  4. **Casos de uso ideales**: Generación de diálogos, asistentes virtuales, sistemas de respuesta automática y  │
│  aplicaciones que requieren contextos prolongados o interacciones dinámicas.                                    │
│                                                                                                                 │
│  # LANGGRAPH                                                                                                    │
│                                                                                                                 │
│  1. **Propósito principal**: LangGraph se enfoca en visualizar y analizar el impacto de las interacciones       │
│  dentro de configuraciones de LLMs, proporcionando herramientas de análisis gráfico y métricas de rendimiento.  │
│                                                                                                                 │
│  2. **Componentes clave**: Tiene generadores de grafos, métricas de rendimiento y herramientas de               │
│  visualización, permitiendo al usuario observar relaciones y flujos de datos en sus aplicaciones de IA.         │
│                                                                                                                 │
│  3. **Modelo de ejecución**: Usa un enfoque de modelado de grafos, donde las relaciones entre componentes son   │
│  representadas como nodos y aristas, facilitando el análisis de flujo y dependencia de datos.                   │
│                                                                                                                 │
│  4. **Casos de uso ideales**: Análisis de modelos de IA, optimización de flujos de trabajo en LLMs, y entornos  │
│  de investigación donde el modelado visual de la interacción entre componentes es crucial.                      │
│                                                                                                                 │
│  # CREWAI                                              

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Redactor Tecnico Senior                                                                                 │
│                                                                                                                 │
│  Task: Redacta un informe ejecutivo integrando la investigacion y el analisis. El informe va dirigido a un CTO  │
│  tecnico que debe elegir un framework. Estructura: (1) resumen de 3 puntos clave, (2) comparativa resumida,     │
│  (3) recomendacion final justificada con el framework ganador y por que.                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Redactor Tecnico Senior                                                                                 │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### Informe Ejecutivo: Evaluación de Frameworks de IA para CTO                                                 │
│                                                                                                                 │
│  #### Resumen de puntos clave                                                                                   │
│                                                                                                                 │
│  1. **LangChain como solución robusta para interacciones LLM**: Diseñado para integrar LLMs con diversas        │
│  fuentes de datos, LangChain ofrece un modelo de ejecución modular que permite decisiones dinámicas y una       │
│  gestión efectiva del contexto, siendo ideal para aplicaciones que requieren interacciones complejas.           │
│                                                                                                                 │
│  2. **LangGraph enfocado en el análisis visual**: Con un enfoque en la visualización de interacciones dentro    │
│  de configuraciones de LLMs, LangGraph es útil para métricas de rendimiento y modelado de relaciones, aunque    │
│  se limita a entornos de investigación y análisis.                                                              │
│                                                                                                                 │
│  3. **CrewAI optimiza la colaboración en proyectos**: Destinado a facilitar el trabajo en equipo en el          │
│  desarrollo de aplicaciones de IA, CrewAI sobresale en entornos colaborativos permitiendo que múltiples         │
│  expertos integren sus habilidades y conocimientos de manera fluida.                                            │
│                                                                                                                 │
│  #### Comparativa resumida                                                                                      │
│                                                                                                                 │
│  | Criterio                  | LangChain                               | LangGraph                              │
│  | CrewAI                                  |                                                                    │
│  |---------------------------|-----------------------------------------|--------------------------------------  │
│  ---|-----------------------------------------|                                                                 │
│  | Curva de aprendizaje       | Moderada; requiere comprensión de LLMs  | Moderada; centrada en visualización   │
│  | Baja; diseño intuitivo y colaborativo   |                                                                    │
│  | Control del flujo          | Alto; diseño modular y progresivo       | Medio; basado en relaciones de nodos  │
│  | Alto; colaboración fluida entre equipos   |                                                                  │
│  | Soporte multi-agente       | Bajo; enfocado en interacción individual  | Bajo; no diseñado para              │
│  multi-agente      | Alto; facilita el trabajo en equipo      |                                                 │
│  | RAG (Razonamiento Auto-Generado) | Alto; permite decisiones dinámicas   | Bajo; enfocado en análisis de      │
│  datos      | Moderado; depende del diseño de tareas   

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Redacta un informe ejecutivo integrando la investigacion y el analisis. El informe va dirigido a un CTO  │
│  tecnico que debe elegir un framework. Estructura: (1) resumen de 3 puntos clave, (2) comparativa resumida,     │
│  (3) recomendacion final justificada con el framework ganador y por que.                                        │
│  Agent: Redactor Tecnico Senior                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


RESULTADO FINAL DEL CREW:
### Informe Ejecutivo: Evaluación de Frameworks de IA para CTO

#### Resumen de puntos clave

1. **LangChain como solución robusta para interacciones LLM**: Diseñado para integrar LLMs con diversas fuentes de datos, LangChain ofrece un modelo de ejecución modular que permite decisiones dinámicas y una gestión efectiva del contexto, siendo ideal para aplicaciones que requieren interacciones complejas.

2. **LangGraph enfocado en el análisis visual**: Con un enfoque en la visualización de interacciones dentro de configuraciones de LLMs, LangGraph es útil para métricas de rendimiento y modelado de relaciones, aunque se limita a entornos de investigación y análisis.

3. **CrewAI optimiza la colaboración en proyectos**: Destinado a facilitar el trabajo en equipo en el desarrollo de aplicaciones de IA, CrewAI sobresale en entornos colaborativos permitiendo que múltiples expertos integren sus habilidades y conocimientos de manera fluida.

#### Comparativa resumida



╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

---
# PARTE 9 — CrewAI: Herramientas Personalizadas

CrewAI usa `BaseTool` para herramientas propias. También acepta directamente herramientas de LangChain definidas con `@tool`.

In [43]:
from crewai.tools import BaseTool
from pydantic import BaseModel, Field as PField
from typing import Type

titulo("CrewAI -- Herramientas con BaseTool", CA)

# Las herramientas de CrewAI son clases que heredan de BaseTool.
# name y description son lo que el agente ve para decidir cuando usar la herramienta.
# args_schema define y valida los argumentos con Pydantic.
# _run() es la implementacion real.

class ArgsCalc(BaseModel):
    expresion: str = PField(description="Expresion matematica. Ej: '2**10 + sqrt(144)'")

class CalculadoraCrewAI(BaseTool):
    name       : str = "calculadora"
    description: str = (
        "Evalua expresiones matematicas. Acepta: +,-,*,/,**,%, sqrt, log, sin, cos, pi."
    )
    args_schema: Type[BaseModel] = ArgsCalc

    def _run(self, expresion: str) -> str:
        import math
        entorno = {"__builtins__": {}, "sqrt":math.sqrt, "log":math.log,
                   "sin":math.sin, "cos":math.cos, "pi":math.pi, "abs":abs, "round":round}
        try:
            return str(round(eval(expresion, entorno), 6))
        except Exception as e:
            return f"Error: {e}"


class ArgsBusqueda(BaseModel):
    query: str = PField(description="Termino a buscar sobre frameworks de IA")

class BuscadorCrewAI(BaseTool):
    name       : str = "buscador_ia"
    description: str = "Busca informacion tecnica sobre LangChain, LangGraph, CrewAI y agentes de IA."
    args_schema: Type[BaseModel] = ArgsBusqueda

    def _run(self, query: str) -> str:
        info = {
            "langchain" : "LangChain: LCEL con |. Maduro, 80K+ stars. Mejor para RAG y conectores.",
            "langgraph" : "LangGraph: StateGraph + TypedDict. Soporta ciclos. Mejor para flujos complejos.",
            "crewai"    : "CrewAI: Agent+Task+Crew. Proceso sequential/hierarchical. Mejor para multi-agente.",
            "react"     : "ReAct: Pensamiento+Accion+Observacion. Yao et al. 2022. Reduce alucinaciones.",
        }
        for k, v in info.items():
            if k in query.lower():
                return v
        return f"Info sobre '{query}': tema del ecosistema de agentes de IA."


calc_crew = CalculadoraCrewAI()
busq_crew = BuscadorCrewAI()

# Test rapido
print(f"Calculadora: {calc_crew._run('sqrt(256) + 2**8')}")
print(f"Buscador   : {busq_crew._run('langgraph ciclos')}")

# Agente con herramientas propias de CrewAI
agente_tools = Agent(
    role="Asistente Tecnico con Herramientas",
    goal="Responder preguntas tecnicas usando busqueda y calculos precisos.",
    backstory="Asistente tecnico con acceso a busqueda de informacion y calculadora.",
    llm="gpt-4o-mini", # Changed to model name string
    tools=[busq_crew, calc_crew],   # herramientas propias de CrewAI
    verbose=True,
)

tarea_tools = Task(
    description=(
        "Busca informacion sobre LangGraph. "
        "Luego calcula: si LangGraph tiene 2 nodos principales (LLM y Tools), "
        "cuanto es 2**8 + sqrt(64)? Interpreta el resultado."
    ),
    expected_output="Descripcion de LangGraph y el resultado del calculo con interpretacion.",
    agent=agente_tools,
)

crew_tools = Crew(
    agents=[agente_tools], tasks=[tarea_tools],
    process=Process.sequential, verbose=False,
)

res_tools = crew_tools.kickoff()
print(f"\n{CA}Resultado:{RST} {res_tools}")


  CrewAI -- Herramientas con BaseTool
Calculadora: 272.0
Buscador   : LangGraph: StateGraph + TypedDict. Soporta ciclos. Mejor para flujos complejos.


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Asistente Tecnico con Herramientas                                                                      │
│                                                                                                                 │
│  Task: Busca informacion sobre LangGraph. Luego calcula: si LangGraph tiene 2 nodos principales (LLM y Tools),  │
│  cuanto es 2**8 + sqrt(64)? Interpreta el resultado.                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool buscador_ia executed with result: LangGraph: StateGraph + TypedDict. Soporta ciclos. Mejor para flujos complejos....
Tool calculadora executed with result: 264.0...


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Asistente Tecnico con Herramientas                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  LangGraph es una herramienta que combina un StateGraph con TypedDict, permitiendo la representación de ciclos  │
│  y siendo más adecuada para flujos de trabajo complejos.                                                        │
│                                                                                                                 │
│  El resultado del cálculo \(2^8 + \sqrt{64}\) es 264.                                                           │
│                                                                                                                 │
│  La interpretación de este resultado es que, al establecer dos nodos principales (LLM y Tools) en LangGraph,    │
│  se puede visualizar la complejidad de un sistema con muchas configuraciones (256) más 8, que podría            │
│  representar elementos adicionales o interacciones, sumando un total de 264 posibilidades o configuraciones en  │
│  un entorno más complejo.                                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


Resultado: LangGraph es una herramienta que combina un StateGraph con TypedDict, permitiendo la representación de ciclos y siendo más adecuada para flujos de trabajo complejos.

El resultado del cálculo \(2^8 + \sqrt{64}\) es 264. 

La interpretación de este resultado es que, al establecer dos nodos principales (LLM y Tools) en LangGraph, se puede visualizar la complejidad de un sistema con muchas configuraciones (256) más 8, que podría representar elementos adicionales o interacciones, sumando un total de 264 posibilidades o configuraciones en un entorno más complejo.


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

---
# PARTE 10 — CrewAI: Proceso Jerárquico

En `Process.hierarchical`, un **manager LLM** coordina al equipo: decide qué agente trabaja en cada momento, verifica la calidad y solicita correcciones si es necesario.

In [46]:
titulo("CrewAI -- Proceso jerarquico (manager LLM)", CA)

# En Process.hierarchical:
# 1. El manager LLM recibe la tarea global.
# 2. Descompone la tarea en subtareas asignadas a agentes.
# 3. Verifica la calidad de cada resultado.
# 4. Pide correcciones si el resultado es insuficiente.
# 5. Sintetiza la respuesta final.

llm_manager = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

inv_h = Agent(
    role="Investigador de Tecnologia de IA",
    goal="Investigar y documentar con precision caracteristicas de frameworks de IA.",
    backstory="Investigador especializado en el ecosistema de LLMs y agentes.",
    llm="gpt-4o-mini", # Changed to model name string
    tools=[BuscadorCrewAI()],
    verbose=False,
)
ana_h = Agent(
    role="Analista de Metricas y Comparativas",
    goal="Analizar datos y calcular metricas comparativas de frameworks.",
    backstory="Analista con experiencia en benchmarking de sistemas de IA.",
    llm="gpt-4o-mini", # Changed to model name string
    tools=[CalculadoraCrewAI()],
    verbose=False,
)
red_h = Agent(
    role="Redactor de Documentacion Tecnica",
    goal="Redactar documentacion clara y accionable para audiencias tecnicas.",
    backstory="Redactor tecnico con experiencia en documentacion de proyectos open source.",
    llm="gpt-4o-mini", # Changed to model name string
    verbose=False,
)

# En jerarquico: la tarea es de ALTO NIVEL
# El manager la descompone y asigna partes a cada agente
tarea_jerarquica = Task(
    description=(
        "Crea un reporte ejecutivo completo sobre como elegir entre LangChain, LangGraph y CrewAI "
        "para tres escenarios: (1) sistema RAG para documentos internos, "
        "(2) agente con estado persistente y reflexion, "
        "(3) equipo de agentes con roles especializados. "
        "El reporte debe incluir recomendacion justificada para cada escenario."
    ),
    expected_output=(
        "Reporte ejecutivo de 400-500 palabras con los tres escenarios analizados, "
        "recomendacion para cada uno y conclusion general."
    ),
    # En jerarquico NO se asigna agent: el manager decide quien hace que
)

crew_jerarquico = Crew(
    agents=[inv_h, ana_h, red_h],
    tasks=[tarea_jerarquica],
    process=Process.hierarchical,      # manager coordina automaticamente
    manager_llm="gpt-4o-mini",           # LLM que actua como manager del equipo (changed to model name string)
    verbose=True,
)

print("Ejecutando crew jerarquico...\n")
res_j = crew_jerarquico.kickoff()
print(f"\n{CA}{BOLD}RESULTADO DEL CREW JERARQUICO:{RST}")
print(res_j)



  CrewAI -- Proceso jerarquico (manager LLM)
Ejecutando crew jerarquico...



╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 1a15d74b-648c-4a6b-80d8-61dfe030070e                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Crea un reporte ejecutivo completo sobre como elegir entre LangChain, LangGraph y CrewAI para tres       │
│  escenarios: (1) sistema RAG para documentos internos, (2) agente con estado persistente y reflexion, (3)       │
│  equipo de agentes con roles especializados. El reporte debe incluir recomendacion justificada para cada        │
│  escenario.                                                                                                     │
│  ID: 5d3a56fb-99f5-42f4-97d0-71cecd5a8ece                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Task: Crea un reporte ejecutivo completo sobre como elegir entre LangChain, LangGraph y CrewAI para tres       │
│  escenarios: (1) sistema RAG para documentos internos, (2) agente con estado persistente y reflexion, (3)       │
│  equipo de agentes con roles especializados. El reporte debe incluir recomendacion justificada para cada        │
│  escenario.                                                                                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: delegate_work_to_coworker                                                                                │
│  Args: {'task': 'Crea un reporte ejecutivo completo sobre como elegir entre LangChain, LangGraph y CrewAI para  │
│  tres escenarios: (1) sistema RAG para documentos internos, (2) agente con estado persistente y r...            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────── ✅ Tool Execution Completed (#1) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: delegate_work_to_coworker                                                                                │
│  Output: # Reporte Ejecutivo: Análisis Comparativo de LangChain, LangGraph y CrewAI                             │
│                                                                                                                 │
│  ## Introducción                                                                                                │
│                                                                                                                 │
│  Este reporte evalúa tres soluciones de inteligencia artificial: LangChain, LangGraph y CrewAI, en función de   │
│  su idoneidad para tres escenarios específicos: un sistema de Recuperación Aumentada Generativa (RAG) para      │
│  documentos internos, un agente con estado persistente y reflexión, y un equipo de agentes con roles            │
│  especializados. Se brindará una recomendación justificada para cada caso a fin de facilitar la toma de         │
│  decisiones.                                                                                                    │
│                                                                                                                 │
│  ## Escenario 1: Sistema RAG para Documentos Internos                                                           │
│                                                                                                                 │
│  ### Análisis                                                                                                   │
│                                                                                                                 │
│  **LangChain** es conocido por su capacidad de integrar múltiples fuentes de datos y su enfoque en la creación  │
│  de flujos de trabajo complejos que implican la utilización de modelos de lenguaje. Su arquitectura modular     │
│  facilita la creación de sistemas RAG altamente personalizados que pueden gestionar grandes volúmenes de        │
│  información y generar respuestas basadas en consultas específicas.                                             │
│                                                                                                                 │
│  **LangGraph**, por su parte, se especializa en la creación de grafos de conocimiento que permiten relacionar   │
│  datos de diferentes documentos. Su ventaja radica en la habilidad para enlazar información relevante de        │
│  manera más visual y comprensible, lo que puede ser útil para la identificación de relaciones entre diferentes  │
│  documentos internos.                                                                                           │
│                                                                                                                 │
│  **CrewAI**, en cambio, se enfoca en la colaboración entre agentes y, aunque puede ser utilizado para           │
│  facilitar la búsqueda de información, carece de las características robustas de comprensión de documentos de   │
│  LangChain y LangGraph.                                                                                         │
│                                                                                                                 │
│  ### Recomendación                                                                                              │
│                                                                                                                 │
│  **LangChain** es la opción recomendada para un sistema

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Reporte Ejecutivo: Análisis Comparativo de LangChain, LangGraph y CrewAI                                     │
│                                                                                                                 │
│  ## Introducción                                                                                                │
│                                                                                                                 │
│  Este reporte evalúa tres soluciones de inteligencia artificial: LangChain, LangGraph y CrewAI, en función de   │
│  su idoneidad para tres escenarios específicos: un sistema de Recuperación Aumentada Generativa (RAG) para      │
│  documentos internos, un agente con estado persistente y reflexión, y un equipo de agentes con roles            │
│  especializados. Se brindará una recomendación justificada para cada caso a fin de facilitar la toma de         │
│  decisiones.                                                                                                    │
│                                                                                                                 │
│  ## Escenario 1: Sistema RAG para Documentos Internos                                                           │
│                                                                                                                 │
│  ### Análisis                                                                                                   │
│                                                                                                                 │
│  **LangChain** es conocido por su capacidad de integrar múltiples fuentes de datos y su enfoque en la creación  │
│  de flujos de trabajo complejos que implican la utilización de modelos de lenguaje. Su arquitectura modular     │
│  facilita la creación de sistemas RAG altamente personalizados que pueden gestionar grandes volúmenes de        │
│  información y generar respuestas basadas en consultas específicas.                                             │
│                                                                                                                 │
│  **LangGraph**, por su parte, se especializa en la creación de grafos de conocimiento que permiten relacionar   │
│  datos de diferentes documentos. Su ventaja radica en la habilidad para enlazar información relevante de        │
│  manera más visual y comprensible, lo que puede ser útil para la identificación de relaciones entre diferentes  │
│  documentos internos.                                                                                           │
│                                                                                                                 │
│  **CrewAI**, en cambio, se enfoca en la colaboración entre agentes y, aunque puede ser utilizado para           │
│  facilitar la búsqueda de información, carece de las características robustas de comprensión de documentos de   │
│  LangChain y LangGraph.                                                                                         │
│                                                                                                                 │
│  ### Recomendación                                                                                              │
│                                                        

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Crea un reporte ejecutivo completo sobre como elegir entre LangChain, LangGraph y CrewAI para tres       │
│  escenarios: (1) sistema RAG para documentos internos, (2) agente con estado persistente y reflexion, (3)       │
│  equipo de agentes con roles especializados. El reporte debe incluir recomendacion justificada para cada        │
│  escenario.                                                                                                     │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


RESULTADO DEL CREW JERARQUICO:
# Reporte Ejecutivo: Análisis Comparativo de LangChain, LangGraph y CrewAI

## Introducción

Este reporte evalúa tres soluciones de inteligencia artificial: LangChain, LangGraph y CrewAI, en función de su idoneidad para tres escenarios específicos: un sistema de Recuperación Aumentada Generativa (RAG) para documentos internos, un agente con estado persistente y reflexión, y un equipo de agentes con roles especializados. Se brindará una recomendación justificada para cada caso a fin de facilitar la toma de decisiones.

## Escenario 1: Sistema RAG para Documentos Internos

### Análisis

**LangChain** es conocido por su capacidad de integrar múltiples fuentes de datos y su enfoque en la creación de flujos de trabajo complejos que implican la utilización de modelos de lenguaje. Su arquitectura modular facilita la creación de sistemas RAG altamente personalizados que pueden gestionar grandes volúmenes de información y generar respuestas basadas en consultas e

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

---
# PARTE 11 — Comparativa Final

In [54]:
# ── TABLA Y GUIA FINAL ───────────────────────────────────────
tabla = '''
=============================================================================
Criterio              | LangChain        | LangGraph         | CrewAI
=============================================================================
Abstraccion           | Media            | Baja (control)    | Alta
Curva aprendizaje     | Media            | Alta              | Baja
Control del flujo     | Limitado         | Total             | Automatico
Ciclos/loops          | No directo       | Si (nativo)       | Via jerarquico
Multi-agente          | Manual           | Como grafo        | Nativo declarativo
RAG                   | Excelente        | Si (usa LC)       | Limitado
Herramientas          | 100+ conectores  | Hereda de LC      | Propio + LC
Estado persistente    | Memory classes   | Checkpointer      | Automatico
Depuracion            | Media            | Alta (por nodo)   | Limitada
Composicion           | LCEL con |       | Grafo tipado      | roles + tasks
Mejor para            | RAG, cadenas     | Flujos complejos  | Equipos de roles
=============================================================================
'''
print(tabla)

guia = '''
CUANDO USAR CADA FRAMEWORK:
──────────────────────────────────────────────────────────────────────

USA LangChain cuando:
  Pipeline RAG (cargar docs -> embeddings -> vectorstore -> LLM).
  Necesitas conectores a fuentes de datos (PDF, SQL, APIs, NoSQL).
  Cadenas lineales con LCEL: paso1 | paso2 | paso3.
  Quieres el ecosistema mas maduro con mas documentacion y comunidad.

USA LangGraph cuando:
  El agente necesita ciclos: reintentar, reflexionar, corregir errores.
  Quieres control total sobre cada paso del flujo de ejecucion.
  Necesitas estado persistente entre sesiones (checkpointer).
  Flujos con bifurcaciones condicionales complejas.
  Quieres depurar nodo a nodo y ver exactamente el estado en cada punto.

USA CrewAI cuando:
  Quieres definir un equipo de agentes con roles especializados.
  La tarea se descompone en roles: investigador, analista, redactor, critico.
  Prefieres sintaxis declarativa (definir QUE, no COMO).
  Necesitas un manager que coordine al equipo automaticamente.
  Prototipado rapido de sistemas multi-agente.

PATRON DE PRODUCCION (los tres juntos):
  LangGraph  -> orquestacion y flujo principal del agente
  LangChain  -> RAG, conectores y herramientas del ecosistema
  CrewAI     -> subtareas que requieren colaboracion de roles especializados
'''
print(guia)


Criterio              | LangChain        | LangGraph         | CrewAI
Abstraccion           | Media            | Baja (control)    | Alta
Curva aprendizaje     | Media            | Alta              | Baja
Control del flujo     | Limitado         | Total             | Automatico
Ciclos/loops          | No directo       | Si (nativo)       | Via jerarquico
Multi-agente          | Manual           | Como grafo        | Nativo declarativo
RAG                   | Excelente        | Si (usa LC)       | Limitado
Herramientas          | 100+ conectores  | Hereda de LC      | Propio + LC
Estado persistente    | Memory classes   | Checkpointer      | Automatico
Depuracion            | Media            | Alta (por nodo)   | Limitada
Composicion           | LCEL con |       | Grafo tipado      | roles + tasks
Mejor para            | RAG, cadenas     | Flujos complejos  | Equipos de roles


CUANDO USAR CADA FRAMEWORK:
──────────────────────────────────────────────────────────────────────

USA Lan